# Data Profiling: FHV - NYC TLC

**Tipo de vehiculo:** For-Hire Vehicle (Taxis negros/limusinas tradicionales)

Este notebook realiza un perfil exploratorio completo del dataset FHV del NYC Taxi and Limousine Commission.
Se analizan estadisticas descriptivas, distribuciones temporales, zonas de pickup y patrones operativos.

**Columnas disponibles:** `dispatching_base_num`, `pickup_datetime`, `dropOff_datetime`, `PUlocationID`, `DOlocationID`, `SR_Flag`, `Affiliated_base_number`

In [ ]:
# Configuracion del entorno y librerias
import sys
sys.path.insert(0, '/home/jovyan/work')

import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns
import pandas as pd
import numpy as np

from pyspark.sql import functions as F
from src.spark_utils import get_spark
from src.paths import RAW

# Configuracion de estilo de graficos
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

VEHICLE = 'fhv'
RAW_PATH = str(RAW[VEHICLE])
print(f'Raw path: {RAW_PATH}')

## 1. Inicializacion de Spark

In [ ]:
# Iniciar sesion de Spark y cargar datos crudos
spark = get_spark(app_name='FHV-Profiling')

# Leer todos los archivos parquet del directorio FHV
df = spark.read.parquet(RAW_PATH)

# Total de registros
total_registros = df.count()
print(f'Total de registros cargados: {total_registros:,}')
print()
print('Schema del dataset FHV:')
df.printSchema()

## 2. Perfil General: Estadisticas Descriptivas

In [ ]:
# Estadisticas descriptivas generales
print('=== Estadisticas Descriptivas ===')
stats_pd = df.describe().toPandas().set_index('summary')
print(stats_pd.to_string())

print()
print('=== Conteo de Nulos por Columna ===')

# Calcular nulos y porcentaje por columna
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).toPandas()

null_df = null_counts.T.rename(columns={0: 'nulos'})
null_df['porcentaje'] = (null_df['nulos'] / total_registros * 100).round(2)
null_df = null_df.sort_values('porcentaje', ascending=False)
print(null_df.to_string())

## 3. Distribucion Temporal: Viajes por Ano y Mes

In [ ]:
# Extraer anio y mes de pickup_datetime
df_temporal = df.withColumn('anio_mes', F.date_format('pickup_datetime', 'yyyy-MM'))

viajes_por_mes = df_temporal.groupBy('anio_mes') \
    .agg(F.count('*').alias('total_viajes')) \
    .orderBy('anio_mes') \
    .toPandas()

# Filtrar registros con fecha valida
viajes_por_mes = viajes_por_mes.dropna(subset=['anio_mes'])
viajes_por_mes = viajes_por_mes[
    viajes_por_mes['anio_mes'].str.match(r'^\d{4}-\d{2}$', na=False)
].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(
    range(len(viajes_por_mes)),
    viajes_por_mes['total_viajes'],
    color='steelblue',
    edgecolor='white',
    linewidth=0.5
)
ax.set_xticks(range(len(viajes_por_mes)))
ax.set_xticklabels(viajes_por_mes['anio_mes'], rotation=90, fontsize=7)
ax.set_xlabel('Periodo (Anio-Mes)')
ax.set_ylabel('Total de Viajes')
ax.set_title('Distribucion Temporal de Viajes FHV por Anio y Mes')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()
print(f'Periodos disponibles: {viajes_por_mes["anio_mes"].min()} a {viajes_por_mes["anio_mes"].max()}')

## 4. Top 10 Bases Operadoras (dispatching_base_num)

In [ ]:
# Top 10 bases operadoras por volumen de viajes
top_bases = df.groupBy('dispatching_base_num') \
    .agg(F.count('*').alias('total_viajes')) \
    .orderBy(F.desc('total_viajes')) \
    .limit(10) \
    .toPandas()

top_bases['dispatching_base_num'] = top_bases['dispatching_base_num'].fillna('(Sin Base)')

fig, ax = plt.subplots(figsize=(10, 6))
colores = sns.color_palette('muted', n_colors=len(top_bases))
bases_rev = top_bases['dispatching_base_num'][::-1].values
viajes_rev = top_bases['total_viajes'][::-1].values
ax.barh(bases_rev, viajes_rev, color=colores)
ax.set_xlabel('Total de Viajes')
ax.set_ylabel('Base Operadora (dispatching_base_num)')
ax.set_title('Top 10 Bases Operadoras - FHV')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, v in enumerate(viajes_rev):
    ax.text(v * 1.01, i, f'{v:,}', va='center', fontsize=8)
plt.tight_layout()
plt.show()
print('Top 10 bases operadoras FHV:')
print(top_bases.to_string(index=False))

## 5. Distribucion por Zonas (Pickup)

In [ ]:
# Top 10 zonas de pickup por volumen de viajes
# Nota: columna en minusculas PUlocationID para FHV
top_zonas = df.groupBy('PUlocationID') \
    .agg(F.count('*').alias('total_viajes')) \
    .filter(F.col('PUlocationID').isNotNull()) \
    .orderBy(F.desc('total_viajes')) \
    .limit(10) \
    .toPandas()

top_zonas['PUlocationID'] = top_zonas['PUlocationID'].astype(str)

fig, ax = plt.subplots(figsize=(10, 6))
colores = sns.color_palette('Blues_d', n_colors=len(top_zonas))
ax.bar(top_zonas['PUlocationID'], top_zonas['total_viajes'], color=colores)
ax.set_xlabel('Zona de Pickup (PUlocationID)')
ax.set_ylabel('Total de Viajes')
ax.set_title('Top 10 Zonas de Pickup - FHV')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, v in enumerate(top_zonas['total_viajes']):
    ax.text(i, v * 1.01, f'{v:,}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()
print('Top 10 zonas de pickup:')
print(top_zonas.to_string(index=False))

## 6. Distribucion Horaria de Viajes

In [ ]:
# Distribucion horaria: extraer hora de pickup_datetime
viajes_por_hora = df.withColumn('hora', F.hour('pickup_datetime')) \
    .filter(F.col('hora').isNotNull()) \
    .groupBy('hora') \
    .agg(F.count('*').alias('total_viajes')) \
    .orderBy('hora') \
    .toPandas()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    viajes_por_hora['hora'],
    viajes_por_hora['total_viajes'],
    marker='o', linewidth=2, color='steelblue', markersize=5
)
ax.fill_between(viajes_por_hora['hora'], viajes_por_hora['total_viajes'], alpha=0.2, color='steelblue')
ax.set_xlabel('Hora del Dia (0-23)')
ax.set_ylabel('Total de Viajes')
ax.set_title('Distribucion Horaria de Viajes FHV')
ax.set_xticks(range(0, 24))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()
hora_pico = int(viajes_por_hora.loc[viajes_por_hora['total_viajes'].idxmax(), 'hora'])
print(f'Hora pico de demanda: {hora_pico:02d}:00 hs')

In [ ]:
# Distribucion de SR_Flag (indicador de viaje compartido)
sr_flag_dist = df.groupBy('SR_Flag') \
    .agg(F.count('*').alias('total')) \
    .toPandas()

sr_flag_dist['SR_Flag'] = sr_flag_dist['SR_Flag'].fillna('Sin Registro').astype(str)

fig, ax = plt.subplots(figsize=(7, 7))
colores_pie = sns.color_palette('pastel', n_colors=len(sr_flag_dist))
wedges, texts, autotexts = ax.pie(
    sr_flag_dist['total'],
    labels=sr_flag_dist['SR_Flag'],
    autopct='%1.2f%%',
    colors=colores_pie,
    startangle=90,
    pctdistance=0.80
)
for t in autotexts:
    t.set_fontsize(9)
ax.set_title('Distribucion de SR_Flag (Viajes Compartidos) - FHV')
plt.tight_layout()
plt.show()
sr_flag_dist['porcentaje'] = (sr_flag_dist['total'] / sr_flag_dist['total'].sum() * 100).round(2)
print('Distribucion SR_Flag:')
print(sr_flag_dist.to_string(index=False))

In [ ]:
# Finalizar sesion de Spark
spark.stop()
print('Profiling completado.')